# Data exploration

Inspect the input layers before trusting anything downstream: coverage, completeness, and the obvious errors that a model will otherwise absorb silently.

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import matplotlib.pyplot as plt

from kajiado_lceo import load_config
from kajiado_lceo.pipeline import run

config = load_config("../config", scenario="baseline")
config.scenario

## Load the layers

Swap the sample generator for `load_inputs(config)` once real data is in `data/raw/`.

In [ ]:
from kajiado_lceo.sample import generate_sample_county

layers = generate_sample_county(seed=42)
buildings = layers["buildings"]
print(f"{len(buildings):,} structures")
buildings.head()

## Footprint area distribution

The bimodality here is what the Table 3.3 area criterion keys on: dwellings and ancillary structures occupy different parts of the distribution.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
buildings["area_m2"].clip(upper=300).hist(bins=60, ax=ax)
ax.axvline(config.get("footprint_filter.min_area_m2"), color="crimson",
           linestyle="--", label="min habitable area")
ax.set_xlabel("Footprint area (m2)"); ax.set_ylabel("Structures")
ax.legend(); plt.show()

## Structure filter attrition

How many structures each criterion removes. Large attrition at one criterion is worth understanding before it becomes a household-count error.

In [ ]:
from kajiado_lceo.preprocessing import filter_footprints, filter_summary

filtered = filter_footprints(buildings, config)
filter_summary(filtered)

## Spatial coverage

Check for gaps that indicate missing tiles rather than empty rangeland.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 8))
ax.scatter(buildings["longitude"], buildings["latitude"], s=1, alpha=0.3)
for lat1, lon1, lat2, lon2 in layers["mv_lines"]:
    ax.plot([lon1, lon2], [lat1, lat2], color="crimson", linewidth=1.2)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_aspect("equal", adjustable="datalim"); plt.show()